# 🤖 UR5e Pick & Place — SAC vs PPO
## Reinforcement Learning với PyBullet (Research Paper)
**Platform:** Google Colab Free (2 vCPU) | **Framework:** Stable-Baselines3 + Gymnasium

---

| Cell | Nội dung |
|------|----------|
| 1 | Setup, Mount Drive, Clone Repo |
| 2 | Custom Gym Environment (UR5eGymEnv) |
| 3 | Optuna Hyperparameter Tuning (tuỳ chọn) |
| 4 | Pipeline Train SAC & PPO × 3 Seeds |
| 5 | Đánh giá, Đồ thị, Video |

> **Colab Free:** 2 vCPU → `n_envs=2` bắt buộc. Dùng `SubprocVecEnv` fallback `DummyVecEnv`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1: SETUP & MOUNT DRIVE                               ║
# ╚══════════════════════════════════════════════════════════════╝

# --- 1.1 Mount Google Drive & tạo thư mục ---
from google.colab import drive
drive.mount('/content/drive')
import os

BASE = '/content/drive/MyDrive/ur5e_rl'
for d in [
    f'{BASE}/models/sac', f'{BASE}/models/ppo', f'{BASE}/models/optuna',
    f'{BASE}/logs/sac',   f'{BASE}/logs/ppo',
    f'{BASE}/norm/sac',   f'{BASE}/norm/ppo',
    f'{BASE}/results',    f'{BASE}/videos',
]:
    os.makedirs(d, exist_ok=True)
print('✅ Drive mounted & folders ready')

# --- 1.2 Cài đặt dependencies ---
!apt-get install -y xvfb > /dev/null 2>&1
!pip install -q stable-baselines3[extra] pybullet gymnasium \
    optuna imageio imageio-ffmpeg pyvirtualdisplay scipy
print('✅ Packages installed')

# --- 1.3 Clone Github Repo ---
# === HƯỚNG DẪN: ===
# Nếu repo PRIVATE, thay REPO_URL bằng:
#   https://<TOKEN>@github.com/Elsa9999/Do_An_RoBot_V2.git
# Lấy TOKEN: Github → Settings → Developer Settings → Personal Access Tokens
REPO_URL = 'https://github.com/Elsa9999/Do_An_RoBot_V2.git'
PROJECT  = '/content/ur5e_project'

if not os.path.exists(PROJECT):
    !git clone {REPO_URL} {PROJECT}
else:
    print('Repo already cloned, pulling latest...')
    !cd {PROJECT} && git pull

import sys
sys.path.insert(0, PROJECT)

# Verify imports từ repo
from kinematics.forward_kinematics import forward_kinematics
from kinematics.inverse_kinematics import inverse_kinematics
from kinematics.workspace_validator import WorkspaceValidator
from simulation.environment import UR5eEnvironment
from utils.transforms import local_to_world, world_to_local
print('✅ Project imports OK')

# --- 1.4 Seeds & Reproducibility ---
import torch, numpy as np, random

SEEDS = [42, 123, 777]
random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ Không có GPU — sẽ dùng CPU (chậm hơn)')

N_ENVS = 2  # HARD LIMIT cho Colab Free 2 vCPU
print(f'✅ n_envs = {N_ENVS} | Seeds = {SEEDS}')

# --- 1.5 Khởi tạo Virtual Display (xvfb) ---
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()
print('✅ Xvfb virtual display started')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2: CUSTOM GYM ENVIRONMENT — UR5eGymEnv               ║
# ╚══════════════════════════════════════════════════════════════╝

import gymnasium as gym
import numpy as np
import pybullet as p
from gymnasium import spaces
from scipy.spatial.transform import Rotation

from kinematics.forward_kinematics import forward_kinematics
from kinematics.inverse_kinematics import inverse_kinematics
from kinematics.workspace_validator import WorkspaceValidator
from simulation.environment import UR5eEnvironment
from utils.transforms import local_to_world, world_to_local


class UR5eGymEnv(gym.Env):
    """UR5e Pick & Place — Cartesian delta-EE control.
    
    Observation (25D):
        q(6) + dq(6) + ee_pos(3) + ee_quat(4) + obj_pos(3) + rel(3)
    Action (3D):
        delta [dx, dy, dz] * ACTION_SCALE
    Reward:
        Clip [-10, 50]
    """
    metadata = {'render_modes': ['rgb_array']}

    MAX_STEPS      = 200
    PICK_THRESHOLD = 0.05   # m — ngưỡng thành công
    ACTION_SCALE   = 0.03   # m/step
    HOME_POSE      = [0, -1.5708, 1.5708, -1.5708, -1.5708, 0]

    def __init__(self, render_mode='rgb_array'):
        super().__init__()
        # Luôn dùng DIRECT (headless) trên Colab
        self._env       = UR5eEnvironment(gui=False)
        self._validator = WorkspaceValidator()
        self._prev_dist   = None
        self._prev_action = np.zeros(3, dtype=np.float32)
        self._q_current   = list(self.HOME_POSE)
        self._dq_current  = [0.0] * 6
        self._steps  = 0
        self._picked = False

        # Obs 25D: q(6)+dq(6)+ee_pos(3)+ee_quat(4)+obj_pos(3)+rel(3)
        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(25,), dtype=np.float32)
        # Action 3D: delta EE
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(3,), dtype=np.float32)

    # ---- Helpers ----
    def _euler_to_quat(self, euler):
        """Euler ZYX → Quaternion (tránh Gimbal lock)."""
        r = Rotation.from_euler('zyx', [euler[2], euler[1], euler[0]])
        return r.as_quat().astype(np.float32)  # [x,y,z,w]

    def _get_obs(self):
        q  = np.array(self._q_current, dtype=np.float32)
        dq = np.array(self._dq_current, dtype=np.float32)
        fk      = forward_kinematics(self._q_current)
        ee_world = np.array(local_to_world(fk['position']), dtype=np.float32)
        ee_quat  = self._euler_to_quat(fk['euler'])
        obj_pos  = np.array(self._env.get_object_pose()[0], dtype=np.float32)
        rel      = (obj_pos - ee_world).astype(np.float32)
        return np.concatenate([q, dq, ee_world, ee_quat, obj_pos, rel])

    def _compute_reward(self, action):
        """Reward function — clip [-10, 50]."""
        fk       = forward_kinematics(self._q_current)
        ee_world = np.array(local_to_world(fk['position']))
        obj_pos  = np.array(self._env.get_object_pose()[0])
        dist     = float(np.linalg.norm(ee_world - obj_pos))

        # Thành phần reward
        r_dist     = -dist * 2.0                          # phạt khoảng cách
        r_progress = (self._prev_dist - dist) * 5.0       # thưởng tiến bộ
        r_success  = 50.0 if dist < self.PICK_THRESHOLD else 0.0
        r_step     = -0.1                                 # phạt mỗi step

        self._prev_dist = dist
        if dist < self.PICK_THRESHOLD:
            self._picked = True

        # Phạt IK fail / ngoài workspace (-2.0 áp dụng ở step())
        ok, _ = self._validator.is_valid_ee(ee_world.tolist())
        r_ws  = -2.0 if not ok else 0.0

        total = r_dist + r_progress + r_success + r_step + r_ws
        return float(np.clip(total, -10.0, 50.0))

    # ---- Gym interface ----
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._steps       = 0
        self._picked      = False
        self._prev_action = np.zeros(3, dtype=np.float32)
        self._q_current   = list(self.HOME_POSE)
        self._dq_current  = [0.0] * 6
        self._env.reset()
        obs = self._get_obs()
        # rel = obs[22:25], prev_dist = norm(rel)
        self._prev_dist = float(np.linalg.norm(obs[22:25]))
        return obs, {}

    def step(self, action):
        self._steps += 1
        action = np.clip(action, -1.0, 1.0).astype(np.float32)
        dx, dy, dz = action * self.ACTION_SCALE

        fk       = forward_kinematics(self._q_current)
        ee_pos   = list(fk['position'])
        ee_euler = list(fk['euler'])

        # World-frame delta → local target → IK
        ee_world  = local_to_world(ee_pos)
        world_new = [ee_world[0]+dx, ee_world[1]+dy, ee_world[2]+dz]
        pos_safe  = self._validator.clamp_to_workspace(world_new)
        local_new = world_to_local(pos_safe)

        ik_result  = inverse_kinematics(local_new, ee_euler,
                                        q_current=self._q_current)
        ik_success = False
        if ik_result['best'] is not None:
            q_new = ik_result['best']
            self._dq_current = [
                (q_new[i] - self._q_current[i]) * 60.0 for i in range(6)]
            self._q_current = q_new
            self._env.set_joint_positions(self._q_current)
            ik_success = True
        else:
            self._dq_current = [0.0] * 6

        self._env.step(5)  # 5 sim substeps

        reward = self._compute_reward(action)
        # Phạt nặng nếu IK fail
        if not ik_success:
            reward = float(np.clip(reward - 2.0, -10.0, 50.0))

        self._prev_action = action.copy()
        terminated = self._picked
        truncated  = self._steps >= self.MAX_STEPS

        return (self._get_obs(), reward, terminated, truncated,
                {'picked': self._picked, 'ik_ok': ik_success})

    def render(self):
        """Trả về RGB array từ PyBullet camera."""
        w, h = 320, 240
        view = p.computeViewMatrixFromYawPitchRoll(
            cameraTargetPosition=[0.5, 0, 0.5],
            distance=1.2, yaw=45, pitch=-35, roll=0, upAxisIndex=2)
        proj = p.computeProjectionMatrixFOV(
            fov=60, aspect=w/h, nearVal=0.1, farVal=3.0)
        _, _, img, _, _ = p.getCameraImage(
            w, h, viewMatrix=view, projectionMatrix=proj,
            renderer=p.ER_TINY_RENDERER)
        return np.array(img, dtype=np.uint8).reshape(h, w, 4)[:, :, :3]

    def close(self):
        self._env.close()


# --- Verify environment ---
from stable_baselines3.common.env_checker import check_env
_test = UR5eGymEnv()
check_env(_test, warn=True)
_test.close()
print('✅ UR5eGymEnv — check_env passed!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3: OPTUNA HYPERPARAMETER TUNING (Tuỳ chọn)           ║
# ╚══════════════════════════════════════════════════════════════╝
# ⚠️ Cell này TUỲ CHỌN — bỏ qua nếu đã có best_params.json

import optuna, json
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

OPTUNA_STEPS  = 15_000   # steps/trial (ngắn gọn)
OPTUNA_TRIALS = 10


def objective_sac(trial):
    """Hàm objective cho Optuna — tìm HP tốt nhất cho SAC."""
    lr       = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    batch_sz = trial.suggest_categorical('batch_size', [128, 256])
    gamma    = trial.suggest_float('gamma', 0.95, 0.999)
    tau      = trial.suggest_float('tau', 0.001, 0.02)
    try:
        vec_env = DummyVecEnv([lambda: UR5eGymEnv() for _ in range(N_ENVS)])
        model = SAC('MlpPolicy', vec_env,
                    learning_rate=lr, batch_size=batch_sz,
                    tau=tau, gamma=gamma,
                    buffer_size=50_000, learning_starts=1_000,
                    verbose=0, device='auto')
        model.learn(total_timesteps=OPTUNA_STEPS)
        # Đánh giá trên 1 env riêng
        eval_env = UR5eGymEnv()
        mean_r, _ = evaluate_policy(model, eval_env,
                                    n_eval_episodes=5, deterministic=True)
        eval_env.close()
        vec_env.close()
        return float(mean_r)
    except Exception as e:
        print(f'⚠️ Trial {trial.number} failed: {e}')
        return -999.0


# Chạy Optuna
print(f'🔍 Optuna: {OPTUNA_TRIALS} trials × {OPTUNA_STEPS} steps ...')
study = optuna.create_study(direction='maximize')
study.optimize(objective_sac, n_trials=OPTUNA_TRIALS,
               show_progress_bar=True)

BEST_PARAMS = study.best_params
print(f'\n✅ Best params: {BEST_PARAMS}')
print(f'   Best value : {study.best_value:.2f}')

# Lưu ra JSON trên Drive
params_path = f'{BASE}/models/optuna/best_params.json'
with open(params_path, 'w') as f:
    json.dump(BEST_PARAMS, f, indent=2)
print(f'💾 Saved → {params_path}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4: TRAIN SAC & PPO — 3 SEEDS                         ║
# ╚══════════════════════════════════════════════════════════════╝

import json, os, pickle
import numpy as np
from stable_baselines3 import SAC, PPO
from stable_baselines3.common.vec_env import (
    SubprocVecEnv, DummyVecEnv, VecNormalize)
from stable_baselines3.common.callbacks import (
    CheckpointCallback, EvalCallback, StopTrainingOnRewardThreshold,
    CallbackList)
from stable_baselines3.common.monitor import Monitor

SEEDS = [42, 123, 777]
N_ENVS = 2
SAC_STEPS = 300_000
PPO_STEPS = 500_000

# --- Load best params từ Optuna (nếu có), fallback defaults ---
params_file = f'{BASE}/models/optuna/best_params.json'
if os.path.exists(params_file):
    with open(params_file) as f:
        BEST_PARAMS = json.load(f)
    print(f'✅ Loaded Optuna params: {BEST_PARAMS}')
else:
    BEST_PARAMS = {'lr': 3e-4, 'batch_size': 256, 'gamma': 0.99, 'tau': 0.005}
    print(f'⚠️ Dùng default params: {BEST_PARAMS}')


def make_env(seed):
    """Factory tạo env có seed."""
    def _init():
        env = UR5eGymEnv()
        env = Monitor(env)
        env.reset(seed=seed)
        return env
    return _init


def make_vec_env(seed, n_envs=N_ENVS):
    """Tạo VecEnv: thử SubprocVecEnv, fallback DummyVecEnv."""
    env_fns = [make_env(seed + i) for i in range(n_envs)]
    try:
        vec = SubprocVecEnv(env_fns)
        print(f'  ✅ SubprocVecEnv({n_envs})')
    except Exception as e:
        print(f'  ⚠️ SubprocVecEnv failed ({e}), fallback DummyVecEnv')
        vec = DummyVecEnv(env_fns)
    # Bọc VecNormalize
    vec = VecNormalize(vec, norm_obs=True, norm_reward=True,
                       clip_obs=10.0, clip_reward=10.0)
    return vec


def build_callbacks(algo_name, seed):
    """Tạo callbacks: Checkpoint + Eval + StopOnReward."""
    ckpt_dir = f'{BASE}/models/{algo_name}/seed{seed}/ckpt'
    log_dir  = f'{BASE}/logs/{algo_name}/seed{seed}'
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)

    checkpoint_cb = CheckpointCallback(
        save_freq=max(25_000 // N_ENVS, 1),
        save_path=ckpt_dir,
        name_prefix=f'{algo_name}_s{seed}')

    # Eval env riêng (1 env, ko VecNormalize)
    eval_env = DummyVecEnv([make_env(seed + 100)])
    # Dùng VecNormalize cho eval nhưng KHÔNG cập nhật stats
    eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False,
                            training=False, clip_obs=10.0)

    stop_cb = StopTrainingOnRewardThreshold(
        reward_threshold=40.0, verbose=1)

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=f'{BASE}/models/{algo_name}/seed{seed}',
        log_path=log_dir,
        eval_freq=max(5_000 // N_ENVS, 1),
        n_eval_episodes=5,
        deterministic=True,
        callback_after_eval=stop_cb)

    return CallbackList([checkpoint_cb, eval_cb])


# ==================== TRAINING LOOP ====================
for algo_name, AlgoCls, total_steps in [
    ('sac', SAC, SAC_STEPS),
    ('ppo', PPO, PPO_STEPS),
]:
    for seed in SEEDS:
        print(f'\n{"="*60}')
        print(f'🚀 Training {algo_name.upper()} | seed={seed} | {total_steps:,} steps')
        print(f'{"="*60}')

        vec_env   = make_vec_env(seed)
        callbacks = build_callbacks(algo_name, seed)

        # Tạo model với best params
        common_kw = dict(
            policy='MlpPolicy',
            env=vec_env,
            learning_rate=BEST_PARAMS.get('lr', 3e-4),
            gamma=BEST_PARAMS.get('gamma', 0.99),
            verbose=1,
            seed=seed,
            device='auto',
        )

        if algo_name == 'sac':
            model = SAC(
                **common_kw,
                batch_size=BEST_PARAMS.get('batch_size', 256),
                tau=BEST_PARAMS.get('tau', 0.005),
                buffer_size=200_000,
                learning_starts=5_000,
                train_freq=1,
                gradient_steps=1,
            )
        else:  # PPO
            model = PPO(
                **common_kw,
                n_steps=2048,
                batch_size=BEST_PARAMS.get('batch_size', 256),
                n_epochs=10,
                clip_range=0.2,
                ent_coef=0.01,
            )

        # Huấn luyện
        model.learn(
            total_timesteps=total_steps,
            callback=callbacks,
            progress_bar=True)

        # Lưu model + normalizer
        save_dir = f'{BASE}/models/{algo_name}/seed{seed}'
        os.makedirs(save_dir, exist_ok=True)
        model.save(f'{save_dir}/{algo_name}_final')
        vec_env.save(f'{BASE}/norm/{algo_name}/vecnorm_s{seed}.pkl')
        print(f'💾 Saved {algo_name.upper()} seed={seed}')

        vec_env.close()

print('\n✅ Training Pipeline hoàn tất!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5: ĐÁNH GIÁ, ĐỒ THỊ & VIDEO                         ║
# ╚══════════════════════════════════════════════════════════════╝

import json, os, pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from stable_baselines3 import SAC, PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import imageio

SEEDS = [42, 123, 777]
N_EVAL_EPISODES = 100

# ─── 5.1 Hàm đánh giá 1 model ───
def evaluate_model(algo_name, AlgoCls, seed, n_episodes=N_EVAL_EPISODES):
    """Load model + normalizer, chạy n_episodes, trả mean/std/success."""
    model_path = f'{BASE}/models/{algo_name}/seed{seed}/{algo_name}_final.zip'
    norm_path  = f'{BASE}/norm/{algo_name}/vecnorm_s{seed}.pkl'

    # Tạo eval env
    eval_env = DummyVecEnv([lambda: Monitor(UR5eGymEnv())])
    if os.path.exists(norm_path):
        eval_env = VecNormalize.load(norm_path, eval_env)
        eval_env.training = False
        eval_env.norm_reward = False

    model = AlgoCls.load(model_path, env=eval_env, device='auto')

    rewards, successes = [], 0
    for ep in range(n_episodes):
        obs = eval_env.reset()
        ep_r, done = 0.0, False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, dones, infos = eval_env.step(action)
            ep_r += r[0]
            done = dones[0]
            if done and infos[0].get('picked', False):
                successes += 1
        rewards.append(ep_r)

    eval_env.close()
    return {
        'mean': float(np.mean(rewards)),
        'std':  float(np.std(rewards)),
        'success_rate': successes / n_episodes * 100,
        'rewards': rewards,
    }


# ─── 5.2 Đánh giá tất cả seeds ───
results = {}
for algo_name, AlgoCls in [('sac', SAC), ('ppo', PPO)]:
    results[algo_name] = {}
    for seed in SEEDS:
        print(f'📊 Evaluating {algo_name.upper()} seed={seed}...')
        results[algo_name][seed] = evaluate_model(algo_name, AlgoCls, seed)
        r = results[algo_name][seed]
        print(f'   Mean={r["mean"]:.2f} ± {r["std"]:.2f} | '
              f'Success={r["success_rate"]:.1f}%')

# Tổng hợp qua 3 seeds
summary = {}
for algo in ['sac', 'ppo']:
    means = [results[algo][s]['mean'] for s in SEEDS]
    srs   = [results[algo][s]['success_rate'] for s in SEEDS]
    summary[algo] = {
        'mean_reward': float(np.mean(means)),
        'std_reward':  float(np.std(means)),
        'mean_sr':     float(np.mean(srs)),
        'std_sr':      float(np.std(srs)),
    }

# Lưu kết quả JSON
with open(f'{BASE}/results/eval_results.json', 'w') as f:
    json.dump({'per_seed': {a: {str(s): v for s, v in sv.items()}
               for a, sv in results.items()}, 'summary': summary},
              f, indent=2)
print(f'\n💾 Results saved → {BASE}/results/eval_results.json')


# ─── 5.3 Vẽ đồ thị (Dark theme) ───
plt.rcParams.update({
    'figure.facecolor': '#1e1e1e',
    'axes.facecolor':   '#2d2d2d',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Subplot 1: Mean Reward vs Timesteps (learning curve) ---
ax1 = axes[0]
colors = {'sac': '#00ccff', 'ppo': '#ff6644'}
for algo_name in ['sac', 'ppo']:
    all_rewards = []
    min_len = float('inf')
    for seed in SEEDS:
        log_dir = f'{BASE}/logs/{algo_name}/seed{seed}'
        if os.path.exists(log_dir):
            try:
                data = np.load(f'{log_dir}/evaluations.npz')
                timesteps = data['timesteps']
                ep_rewards = data['results'].mean(axis=1)
                all_rewards.append(ep_rewards)
                min_len = min(min_len, len(ep_rewards))
            except Exception:
                pass
    if all_rewards and min_len > 0:
        # Cắt về cùng chiều dài
        all_rewards = np.array([r[:min_len] for r in all_rewards])
        timesteps   = timesteps[:min_len]
        mean_r = all_rewards.mean(axis=0)
        std_r  = all_rewards.std(axis=0)
        ax1.plot(timesteps, mean_r, color=colors[algo_name],
                 label=algo_name.upper(), linewidth=2)
        ax1.fill_between(timesteps, mean_r - std_r, mean_r + std_r,
                         color=colors[algo_name], alpha=0.2)

ax1.set_title('Mean Reward vs Timesteps', fontsize=13, fontweight='bold')
ax1.set_xlabel('Timesteps')
ax1.set_ylabel('Mean Reward')
ax1.legend(facecolor='#333', labelcolor='white')
ax1.grid(alpha=0.2)

# --- Subplot 2: Success Rate (Bar chart) ---
ax2 = axes[1]
algos = ['SAC', 'PPO']
sr_means = [summary['sac']['mean_sr'], summary['ppo']['mean_sr']]
sr_stds  = [summary['sac']['std_sr'],  summary['ppo']['std_sr']]
bars = ax2.bar(algos, sr_means, yerr=sr_stds, capsize=8,
               color=['#00ccff', '#ff6644'], edgecolor='white',
               linewidth=1.2, alpha=0.9)
for bar, val in zip(bars, sr_means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold',
             fontsize=12, color='white')
ax2.set_title('Success Rate (%)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Success Rate (%)')
ax2.set_ylim(0, 110)
ax2.grid(axis='y', alpha=0.2)

# --- Subplot 3: Summary Table ---
ax3 = axes[2]
ax3.axis('off')
table_data = [
    ['Metric', 'SAC', 'PPO'],
    ['Mean Reward',
     f'{summary["sac"]["mean_reward"]:.2f} ± {summary["sac"]["std_reward"]:.2f}',
     f'{summary["ppo"]["mean_reward"]:.2f} ± {summary["ppo"]["std_reward"]:.2f}'],
    ['Success Rate',
     f'{summary["sac"]["mean_sr"]:.1f}% ± {summary["sac"]["std_sr"]:.1f}%',
     f'{summary["ppo"]["mean_sr"]:.1f}% ± {summary["ppo"]["std_sr"]:.1f}%'],
    ['Train Steps', f'{300_000:,}', f'{500_000:,}'],
    ['Seeds', '42, 123, 777', '42, 123, 777'],
]
table = ax3.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 1.8)
# Style header row
for j in range(3):
    table[0, j].set_facecolor('#444')
    table[0, j].set_text_props(fontweight='bold', color='white')
for i in range(1, len(table_data)):
    for j in range(3):
        table[i, j].set_facecolor('#333')
        table[i, j].set_text_props(color='white')
ax3.set_title('Summary Table', fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
fig_path = f'{BASE}/results/sac_vs_ppo_results.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Figure saved → {fig_path}')


# ─── 5.4 Xuất Video episode thành công nhất ───
def record_best_episode(algo_name, AlgoCls):
    """Record video của episode có reward cao nhất."""
    # Tìm seed có mean reward cao nhất
    best_seed = max(SEEDS, key=lambda s: results[algo_name][s]['mean'])
    model_path = f'{BASE}/models/{algo_name}/seed{best_seed}/{algo_name}_final.zip'
    norm_path  = f'{BASE}/norm/{algo_name}/vecnorm_s{best_seed}.pkl'

    # Tạo env đơn cho recording
    rec_env = UR5eGymEnv()
    vec_env = DummyVecEnv([lambda: Monitor(rec_env)])
    if os.path.exists(norm_path):
        vec_env = VecNormalize.load(norm_path, vec_env)
        vec_env.training = False
        vec_env.norm_reward = False

    model = AlgoCls.load(model_path, env=vec_env, device='auto')

    # Chạy nhiều episode, giữ lại episode tốt nhất
    best_frames, best_reward = [], -float('inf')
    for trial in range(10):
        obs = vec_env.reset()
        frames, ep_r, done = [], 0.0, False
        while not done:
            # Render frame
            try:
                frame = rec_env.render()
                if frame is not None:
                    frames.append(frame)
            except Exception:
                pass
            action, _ = model.predict(obs, deterministic=True)
            obs, r, dones, infos = vec_env.step(action)
            ep_r += r[0]
            done = dones[0]
        if ep_r > best_reward and len(frames) > 0:
            best_reward = ep_r
            best_frames = frames.copy()

    vec_env.close()

    # Lưu video
    if best_frames:
        vid_path = f'{BASE}/videos/{algo_name}_best_episode.mp4'
        imageio.mimsave(vid_path, best_frames, fps=30)
        print(f'🎬 Video saved → {vid_path} '
              f'({len(best_frames)} frames, reward={best_reward:.2f})')
    else:
        print(f'⚠️ Không render được frames cho {algo_name.upper()}')


print('\n🎬 Recording best episodes...')
for algo_name, AlgoCls in [('sac', SAC), ('ppo', PPO)]:
    record_best_episode(algo_name, AlgoCls)

print('\n✅ HOÀN TẤT! Tất cả kết quả đã lưu trên Google Drive.')
print(f'   📁 {BASE}')